# L07 · 技术指标入门

**预计时长**：60 min | **难度**：⭐⭐⭐ | **前置**：L06

## 本节目标
1. 理解趋势类（MA/EMA）与震荡类（MACD/RSI）指标的本质
2. 用 pandas 实现 MA / EMA / MACD / RSI
3. `np.where` 生成交易信号
4. 对比 qtrader.strategies.DualMAStrategy 的实现

## 第 1 段：金融概念

### 趋势类 vs 震荡类
| 类型 | 代表 | 适合行情 | 缺点 |
|------|------|---------|------|
| 趋势类 | MA / EMA / MACD | 单边行情 | 震荡市频繁假信号 |
| 震荡类 | RSI / KDJ | 震荡行情 | 单边行情"钝化"失效 |

### MA（简单移动平均）
MA_N = 过去 N 日收盘价的算术平均。**等权重**。
- MA5 / MA20 / MA60：短期 / 中期 / 长期
- 金叉（短上穿长）：买入信号
- 死叉（短下穿长）：卖出信号

### EMA（指数移动平均）
EMA_N = 越近的日子权重越大（指数衰减）。
- 比反应迟缓的 MA 更敏感
- 公式：EMA_today = α × price + (1-α) × EMA_yesterday，α = 2/(N+1)

### MACD（Moving Average Convergence Divergence）
- DIF = EMA12 - EMA26（"快线减慢线"）
- DEA = EMA9(DIF)（DIF 的 9 日 EMA，"信号线"）
- Histogram = 2 × (DIF - DEA)（柱状图，红绿）
- 信号：DIF 上穿 DEA → 买；下穿 → 卖

### RSI（Relative Strength Index，相对强弱）
RSI_N = 100 - 100/(1 + RS)，RS = N 日平均涨幅 / N 日平均跌幅
- 取值 0-100
- > 70：超买（可能要跌）
- < 30：超卖（可能要涨）
- N 常用 6 / 12 / 14

In [ ]:
import sys
from pathlib import Path

# 自动定位 phase1_foundation 目录 + project root（兼容两种 jupyter 启动位置）
_cwd = Path.cwd()
_p1 = _cwd if (_cwd / '_data.py').exists() else (_cwd / 'learning' / 'phase1_foundation')
sys.path.insert(0, str(_p1))
_proj = _p1.parent.parent if _p1.name == 'phase1_foundation' else _p1
if (_proj / 'qtrader' / '__init__.py').exists():
    sys.path.insert(0, str(_proj))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import _style
from _data import get_stock_data

In [ ]:
byd = get_stock_data('002594').set_index('date')

## 第 2 段：MA 与 EMA

In [ ]:
# pandas 内置 rolling 实现MA
byd['ma5']  = byd['close'].rolling(5).mean()
byd['ma20'] = byd['close'].rolling(20).mean()

# ewm 实现 EMA（adjust=False 表示用递推公式）
byd['ema12'] = byd['close'].ewm(span=12, adjust=False).mean()
byd['ema26'] = byd['close'].ewm(span=26, adjust=False).mean()

df_plot = byd.tail(180)
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df_plot.index, df_plot['close'], label='close', color='black', alpha=0.6)
ax.plot(df_plot.index, df_plot['ma5'],   label='MA5',  color=_style.COLORS['ma_short'])
ax.plot(df_plot.index, df_plot['ma20'],  label='MA20', color=_style.COLORS['ma_long'])
ax.plot(df_plot.index, df_plot['ema12'], label='EMA12', linestyle='--')
ax.legend()
plt.show()

## 第 3 段：MACD

In [ ]:
byd['dif'] = byd['ema12'] - byd['ema26']
byd['dea'] = byd['dif'].ewm(span=9, adjust=False).mean()
byd['hist'] = 2 * (byd['dif'] - byd['dea'])

df_plot = byd.tail(180)
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True,
                                gridspec_kw={'height_ratios': [2, 1]})
ax1.plot(df_plot.index, df_plot['close'], color='black', alpha=0.6, label='close')
ax1.plot(df_plot.index, df_plot['dif'], label='DIF', color='blue')
ax1.plot(df_plot.index, df_plot['dea'], label='DEA', color='orange')
ax1.legend()

colors = ['red' if v > 0 else 'green' for v in df_plot['hist']]
ax2.bar(df_plot.index, df_plot['hist'], color=colors, width=0.8)
ax2.axhline(0, color='black', alpha=0.3)
ax2.set_title('MACD Histogram')

plt.tight_layout(); plt.show()

## 第 4 段：RSI

In [ ]:
def compute_rsi(prices: pd.Series, period: int = 14) -> pd.Series:
    """RSI 实现（SMA 简化版）。

    注意：通达信/同花顺默认用 Wilder's RSI（用 EMA-like 递推平滑），
    数值上与本函数略有差异。教学版用 SMA 已足够；实战对照行情软件
    时记得这个口径差。
    """
    delta = prices.diff()
    gain = delta.where(delta > 0, 0).rolling(period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(period).mean()
    rs = gain / loss
    return 100 - 100 / (1 + rs)

byd['rsi14'] = compute_rsi(byd['close'], 14)

df_plot = byd.tail(180)
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True,
                                gridspec_kw={'height_ratios': [2, 1]})
ax1.plot(df_plot.index, df_plot['close'], color='black')
ax2.plot(df_plot.index, df_plot['rsi14'], color='purple')
ax2.axhline(70, color='red', linestyle='--', alpha=0.5, label='超买 70')
ax2.axhline(30, color='green', linestyle='--', alpha=0.5, label='超卖 30')
ax2.set_ylim(0, 100); ax2.legend()
plt.tight_layout(); plt.show()

## 第 5 段：`np.where` 生成信号

In [ ]:
# 双均线信号：MA5 > MA20 持仓 1，否则 0
byd['signal'] = np.where(byd['ma5'] > byd['ma20'], 1, 0)
# 关键：shift(1) 避免未来函数
byd['signal_used'] = byd['signal'].shift(1).fillna(0)

byd['strategy_ret'] = byd['signal_used'] * byd['close'].pct_change()
byd['strategy_nav'] = (1 + byd['strategy_ret']).cumprod()
byd['buy_hold_nav'] = byd['close'] / byd['close'].iloc[0]

fig, ax = plt.subplots(figsize=(12, 5))
byd['strategy_nav'].plot(ax=ax, label='MA 双均线策略')
byd['buy_hold_nav'].plot(ax=ax, label='买入持有', alpha=0.6)
ax.legend()
ax.set_title('比亚迪 双均线 vs 买入持有')
plt.show()

# 胜率统计
trade_rets = byd.loc[byd['signal_used'].diff() == 1, 'strategy_ret']
print(f"开仓次数: {len(trade_rets)}, 平均持仓期收益: {trade_rets.sum()/max(len(trade_rets),1)*100:.2f}%")

## 第 6 段：对比 qtrader 框架

In [ ]:
# qtrader 已实现的 DualMA 策略（common_imports 已把 project root 加入 sys.path）
from qtrader.strategies import DualMAStrategy
from qtrader.engine import BacktestEngine

df_qt = get_stock_data('002594')
result = BacktestEngine().run(df_qt, DualMAStrategy(short=5, long=20))
print(f"qtrader DualMA(5,20) 总收益: {result.metrics['total_return']*100:.2f}%")
print(f"策略净值末值: {result.nav.iloc[-1]:.3f}")

看一下 `qtrader/strategies.py` 里 `DualMAStrategy.generate_signals` 的实现——你会发现它就是用 `rolling().mean()` + 比较，**和上面手写的一模一样**。这就是 Phase 2 的入口：你已经会写策略了。

## 第 7 段：随堂小练

### 小练：实现 EMA 交叉策略
用 EMA12 上穿 EMA26 作为买入信号，下穿作为卖出，跑比亚迪 2022-2024 的回测。

In [ ]:
# TODO: 你的代码（约 6 行）

## 第 8 段：课后练习 + 下节预告

### 📝 `exercises/ex07.py`
1. 写 `compute_macd(prices, fast=12, slow=26, signal=9)` 返回 DataFrame（dif/dea/hist 三列）
2. 写 `rsi_signal(prices, period=14, oversold=30, overbought=70)` 返回信号 Series（超卖+1，超买-1，中性 0）
3. 双均线参数搜索：在 (short, long) ∈ {(5,20), (10,30), (5,60), (20,60)} 中找比亚迪最优组合（按夏普）

### 🔮 下节 L08：PE 估值 + 行业对比
价格类指标（MA/MACD）只看"价格本身"，**估值指标（PE）**告诉你"价格相对价值是贵还是便宜"。

## 第 9 段：Jupyter tip 🔧
- `?DualMAStrategy` 查类文档；`??DualMAStrategy` 查源码（双问号）
- `%run my_script.py` 在 notebook 里跑外部脚本，变量自动注入
- `np.where(cond, a, b)` vs `pd.Series.where(cond, other)`：方向相反，注意！